# **Waste Material Segregation for Improving Waste Management**

## **Objective**

The objective of this project is to implement an effective waste material segregation system using convolutional neural networks (CNNs) that categorises waste into distinct groups. This process enhances recycling efficiency, minimises environmental pollution, and promotes sustainable waste management practices.

The key goals are:

* Accurately classify waste materials into categories like cardboard, glass, paper, and plastic.
* Improve waste segregation efficiency to support recycling and reduce landfill waste.
* Understand the properties of different waste materials to optimise sorting methods for sustainability.

## **Data Understanding**

The Dataset consists of images of some common waste materials.

1. Food Waste
2. Metal
3. Paper
4. Plastic
5. Other
6. Cardboard
7. Glass


**Data Description**

* The dataset consists of multiple folders, each representing a specific class, such as `Cardboard`, `Food_Waste`, and `Metal`.
* Within each folder, there are images of objects that belong to that category.
* However, these items are not further subcategorised. <br> For instance, the `Food_Waste` folder may contain images of items like coffee grounds, teabags, and fruit peels, without explicitly stating that they are actually coffee grounds or teabags.

## **1. Load the data**

Load and unzip the dataset zip file.

**Import Necessary Libraries**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
import os
import zipfile

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix


Load the dataset.

## **2. Data Preparation** <font color=red> [25 marks] </font><br>


### **2.1 Load and Preprocess Images** <font color=red> [8 marks] </font><br>

Let us create a function to load the images first. We can then directly use this function while loading images of the different categories to load and crop them in a single step.

#### **2.1.1** <font color=red> [3 marks] </font><br>
Create a function to load the images.

In [ ]:
# Create a function to load the raw images
def load_images_from_folder(folder_path, target_size=(128, 128)):
    images = []
    labels = []
    for label in os.listdir(folder_path):
        label_path = os.path.join(folder_path, label)
        if os.path.isdir(label_path):
            for filename in os.listdir(label_path):
                img_path = os.path.join(label_path, filename)
                try:
                    img = Image.open(img_path).convert('RGB')
                    img = img.resize(target_size)
                    img_array = np.array(img)
                    images.append(img_array)
                    labels.append(label)
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")
    return np.array(images), np.array(labels)


#### **2.1.2** <font color=red> [5 marks] </font><br>
Load images and labels.

Load the images from the dataset directory. Labels of images are present in the subdirectories.

Verify if the images and labels are loaded correctly.

In [ ]:
# Get the images and their labels
data_root_dir = "/content/drive/MyDrive/Colab Notebooks/data_CNN/data" # Corrected path to point to the directory containing class folders
images, labels = load_images_from_folder(data_root_dir)

print(f"Total images loaded: {len(images)}")
print(f"Total labels loaded: {len(labels)}")
print(f"Shape of images array: {images.shape}")
print(f"Sample labels: {labels[:5]}")


Perform any operations, if needed, on the images and labels to get them into the desired format.

### **2.2 Data Visualisation** <font color=red> [9 marks] </font><br>

#### **2.2.1** <font color=red> [3 marks] </font><br>
Create a bar plot to display the class distribution

In [ ]:
# Visualise Data Distribution
plt.figure(figsize=(10, 6))
sns.countplot(y=labels, order=pd.Series(labels).value_counts().index, palette='viridis')
plt.title('Distribution of Waste Material Classes')
plt.xlabel('Count')
plt.ylabel('Waste Material Class')
plt.show()


#### **2.2.2** <font color=red> [3 marks] </font><br>
Visualise some sample images

In [ ]:
# Visualise Sample Images (across different labels)
def visualize_sample_images(images, labels, num_samples_per_class=2):
    unique_labels = np.unique(labels)
    plt.figure(figsize=(15, 10))

    for i, class_label in enumerate(unique_labels):
        class_indices = np.where(labels == class_label)[0]
        sample_indices = np.random.choice(class_indices, min(num_samples_per_class, len(class_indices)), replace=False)

        for j, idx in enumerate(sample_indices):
            plt.subplot(len(unique_labels), num_samples_per_class, i * num_samples_per_class + j + 1)
            plt.imshow(images[idx])
            plt.title(class_label)
            plt.axis('off')
    plt.tight_layout()
    plt.show()

visualize_sample_images(images, labels)


#### **2.2.3** <font color=red> [3 marks] </font><br>
Based on the smallest and largest image dimensions, resize the images.

In [ ]:
# Find the smallest and largest image dimensions from the data set
# Assuming images are already loaded and resized to (128, 128) in the load_images_from_folder function
# If you need to find original dimensions, you would need to load them without resizing first.

# For the current loaded images (already resized to 128x128):
print(f"All loaded images have a fixed height of: {images.shape[1]}")
print(f"All loaded images have a fixed width of: {images.shape[2]}")
print(f"All loaded images have {images.shape[3]} color channels.")

# If we were to calculate from original images, a function like this would be needed:
# def get_image_dimensions(folder_path):
#     heights = []
#     widths = []
#     for label in os.listdir(folder_path):
#         label_path = os.path.join(folder_path, label)
#         if os.path.isdir(label_path):
#             for filename in os.listdir(label_path):
#                 img_path = os.path.join(label_path, filename)
#                 try:
#                     with Image.open(img_path) as img:
#                         widths.append(img.width)
#                         heights.append(img.height)
#                 except Exception as e:
#                     print(f"Error processing image {img_path}: {e}")
#     if heights and widths:
#         print(f"Smallest height: {min(heights)}, Largest height: {max(heights)}")
#         print(f"Smallest width: {min(widths)}, Largest width: {max(widths)}")
#     else:
#         print("No images found or processed.")

# get_image_dimensions(data_root_dir) # Use data_root_dir instead of output_dir


In [ ]:
# Resize the image dimensions
# Images were already resized to (128, 128) during the loading process.
# If a different size is desired, the load_images_from_folder function's target_size parameter can be changed.

TARGET_SIZE = (128, 128) # Defining the target size for consistency

# If you need to re-process images to a new size:
# images, labels = load_images_from_folder(data_root_dir, target_size=NEW_TARGET_SIZE) # Use data_root_dir
# print(f"Images resized to {images.shape[1]}x{images.shape[2]}")

print(f"Images are already consistently sized to {TARGET_SIZE[0]}x{TARGET_SIZE[1]}")


### **2.3 Encoding the classes** <font color=red> [3 marks] </font><br>

There are seven classes present in the data.

We have extracted the images and their labels, and visualised their distribution. Now, we need to perform encoding on the labels. Encode the labels suitably.

####**2.3.1** <font color=red> [3 marks] </font><br>
Encode the target class labels.

In [ ]:
# Encode the labels suitably
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoded_labels = encoder.fit_transform(labels)

# Convert to one-hot encoding for Keras
num_classes = len(encoder.classes_)
one_hot_labels = keras.utils.to_categorical(encoded_labels, num_classes=num_classes)

print(f"Original labels: {labels[:5]}")
print(f"Encoded labels: {encoded_labels[:5]}")
print(f"One-hot encoded labels (first 5 for all classes):\n{one_hot_labels[:5]}")
print(f"Number of unique classes: {num_classes}")
print(f"Class names: {encoder.classes_}")


### **2.4 Data Splitting** <font color=red> [5 marks] </font><br>

#### **2.4.1** <font color=red> [5 marks] </font><br>
Split the dataset into training and validation sets

In [ ]:
# Assign specified parts of the dataset to train and validation sets
from sklearn.model_selection import train_test_split

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(images, one_hot_labels, test_size=0.2, random_state=42, stratify=encoded_labels)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_val: {y_val.shape}")


## **3. Model Building and Evaluation** <font color=red> [20 marks] </font><br>

### **3.1 Model building and training** <font color=red> [15 marks] </font><br>

#### **3.1.1** <font color=red> [10 marks] </font><br>
Build and compile the model. Use 3 convolutional layers. Add suitable normalisation, dropout, and fully connected layers to the model.

Test out different configurations and report the results in conclusions.

In [ ]:
# Build and compile the model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax') # num_classes should be determined from your data
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()


#### **3.1.2** <font color=red> [5 marks] </font><br>
Train the model.

Use appropriate metrics and callbacks as needed.

In [ ]:
# Training
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)

history = model.fit(
    X_train,
    y_train,
    epochs=5, # Increased epochs for better training, early stopping will prevent overfitting
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    batch_size=32
)


### **3.2 Model Testing and Evaluation** <font color=red> [5 marks] </font><br>

#### **3.2.1** <font color=red> [5 marks] </font><br>
Evaluate the model on test dataset. Derive appropriate metrics.

In [ ]:
# Evaluate on the test set; display suitable metrics
# The current split is train/validation. We will evaluate on the validation set as a proxy for a test set.
# For a proper evaluation, a separate test set should have been held out from the initial split.

loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

# Predict probabilities for the validation set
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_val, axis=1)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=encoder.classes_))

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


## **4. Data Augmentation** <font color=red> [optional] </font><br>

#### **4.1 Create a Data Augmentation Pipeline**

##### **4.1.1**
Define augmentation steps for the datasets.

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomTranslation

# Define augmentation steps to augment images
data_augmentation = tf.keras.Sequential([
    RandomFlip('horizontal_and_vertical'),
    RandomRotation(0.2),
    RandomZoom(0.2),
    RandomTranslation(height_factor=0.2, width_factor=0.2)
])

print("Data augmentation pipeline defined.")

Augment and resample the images.
In case of class imbalance, you can also perform adequate undersampling on the majority class and augment those images to ensure consistency in the input datasets for both classes.

Augment the images.

In [ ]:
# Create a function to augment the images
def augment_images(images_array, data_augmentation_model):
    augmented_images = data_augmentation_model(images_array)
    return augmented_images

print("Function 'augment_images' defined.")

In [ ]:
# Create the augmented training dataset
augmented_X_train = data_augmentation(X_train)

print(f"Original X_train shape: {X_train.shape}")
print(f"Augmented X_train shape: {augmented_X_train.shape}")

##### **4.1.2**

Train the model on the new augmented dataset.

In [ ]:
history_augmented = model.fit(
    augmented_X_train,
    y_train,
    epochs=5, # Increased epochs for better training, early stopping will prevent overfitting
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    batch_size=32
)

print("Model training with augmented data complete.")

## **5. Conclusions** <font color = red> [5 marks]</font>

#### **5.1 Conclude with outcomes and insights gained** <font color =red> [5 marks] </font>

* Report your findings about the data
* Report model training results

# Task
Define a data augmentation pipeline using TensorFlow's image preprocessing layers or `ImageDataGenerator` to apply transformations like rotation, flipping, zooming, and shifting to the training images. Apply this defined data augmentation to the training dataset and retrain the CNN model. Ensure to use appropriate callbacks like `EarlyStopping` and `ReduceLROnPlateau`. Finally, evaluate the retrained model's performance on the validation set using metrics such as accuracy, loss, classification report, and confusion matrix, and then compare these results with the previous model's performance without augmentation.

## Define Data Augmentation

### Subtask:
Define a data augmentation pipeline using TensorFlow's image preprocessing layers to apply transformations like rotation, flipping, zooming, and shifting to the training images.
